# LeetCode 393 — UTF-8 Validation bằng Python

Link đề: https://leetcode.com/problems/utf-8-validation/ · Độ khó: **Medium** · Chủ đề: Bit Manipulation, Array

Skeleton LeetCode:

```python
class Solution:
    def validUtf8(self, data: List[int]) -> bool:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu quy tắc mã hóa **UTF-8**: mỗi ký tự dài 1–4 byte, byte đầu cho biết **bao nhiêu byte** thuộc ký tự đó, các byte sau đều bắt đầu bằng bit `10`.
2. Nắm các công cụ Python về bit: số nhị phân `0b...`, `bin()` / `format(v, '08b')`, **dịch bit** `>>`, **mặt nạ** `&`, và so sánh với hằng nhị phân.
3. Đi từ cách **đổi từng byte thành chuỗi 8 ký tự `0/1`** rồi kiểm tra bằng `startswith` → **một lượt với biến đếm `cnt` và phép dịch bit** → **nhảy theo từng ký tự** → **bảng tra 256 phần tử** → **biểu thức chính quy** trên chuỗi bit.
4. Hiểu mấu chốt: chỉ cần nhớ **"còn bao nhiêu byte tiếp theo phải bắt đầu bằng `10`"** — một biến đếm, không cần nhớ gì khác.
5. Tránh các lỗi kinh điển: quên kiểm tra `cnt == 0` ở cuối, không kiểm tra tiền tố `10` của byte tiếp theo, nhận nhầm `11111xxx` hoặc `10xxxxxx` làm byte đầu hợp lệ.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `data` biểu diễn dữ liệu. Trả về `true` nếu đó là một chuỗi byte **UTF-8 hợp lệ**.

Một ký tự UTF-8 dài từ **1 đến 4 byte**:

| Số byte | Dạng nhị phân |
|---|---|
| 1 | `0xxxxxxx` |
| 2 | `110xxxxx 10xxxxxx` |
| 3 | `1110xxxx 10xxxxxx 10xxxxxx` |
| 4 | `11110xxx 10xxxxxx 10xxxxxx 10xxxxxx` |

`x` là bit bất kỳ (`0` hoặc `1`). Mỗi số nguyên chỉ dùng **8 bit thấp nhất** để lưu một byte.

**Ví dụ 1:** `data = [197, 130, 1]` → `true`. Nhị phân: `11000101 10000010 00000001` = ký tự 2 byte rồi ký tự 1 byte.

**Ví dụ 2:** `data = [235, 140, 4]` → `false`. Nhị phân: `11101011 10001100 00000100`. Ba bit đầu `111` rồi `0` ⇒ ký tự 3 byte; byte tiếp theo `10001100` đúng dạng `10xxxxxx`; nhưng byte kế `00000100` **không** bắt đầu bằng `10`, nên sai.

**Ràng buộc:** `1 <= data.length <= 2 * 10^4`, `0 <= data[i] <= 255`.

In [1]:
examples = [
    ([197, 130, 1], True),
    ([235, 140, 4], False),
]
for data, expected in examples:
    print(data, [format(b, '08b') for b in data], "-> kỳ vọng", expected)

[197, 130, 1] ['11000101', '10000010', '00000001'] -> kỳ vọng True
[235, 140, 4] ['11101011', '10001100', '00000100'] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Số nhị phân trong Python: `0b...`, `bin()`, `format(v, '08b')`

Python viết số nhị phân với tiền tố `0b`: `0b110 == 6`. `bin(6)` trả về chuỗi `'0b110'`. Để có **đủ 8 chữ số** (thêm số `0` đứng đầu) dùng `format(v, '08b')` — rất tiện để "nhìn" một byte.

In [2]:
print(0b110, 0b11000101)             # 6 197
print(bin(197))                      # 0b11000101
print(format(197, '08b'))            # 11000101
print(format(1, '08b'))              # 00000001  (bù đủ 8 chữ số)
print(int('11000101', 2))            # 197  (ngược lại: chuỗi bit -> số)

6 197
0b11000101
11000101
00000001
197


### 3.2. Dịch phải `>>`: lấy các bit cao nhất

`v >> k` bỏ đi `k` bit thấp nhất của `v` (dịch sang phải `k` vị trí). Với một byte 8 bit, `v >> 7` còn đúng **bit cao nhất**, `v >> 6` còn **2 bit cao nhất**, `v >> 5` còn **3 bit cao nhất**, `v >> 4` còn 4 bit cao nhất, `v >> 3` còn 5 bit cao nhất. Đây là cách ngắn gọn để hỏi "byte này bắt đầu bằng dãy bit nào?".

In [3]:
v = 0b11000101                       # 197
print(format(v >> 7, 'b'))           # 1       (1 bit cao nhất)
print(format(v >> 6, 'b'))           # 11      (2 bit cao nhất)
print(format(v >> 5, 'b'))           # 110     (3 bit cao nhất)
print(format(v >> 4, 'b'))           # 1100
print(v >> 5 == 0b110)               # True: bắt đầu bằng 110

1
11
110
1100
True


### 3.3. Mặt nạ `&` (bitwise AND)

`v & mask` giữ lại các bit của `v` ở những vị trí mà `mask` có bit `1`, còn lại về `0`. Muốn xem "3 bit cao nhất của `v` có phải `110` không": `v & 0b11100000 == 0b11000000`. Hai cách (`>>` và `&`) tương đương; ta chủ yếu dùng `>>` vì gọn.

In [4]:
v = 0b11000101
print(format(v & 0b11100000, '08b'))                 # 11000000
print((v & 0b11100000) == 0b11000000)                # True
print((v >> 5) == 0b110)                             # True (cùng ý nghĩa)
w = 0b10000010                                        # 130, byte "tiếp theo"
print((w & 0b11000000) == 0b10000000, (w >> 6) == 0b10)    # True True

11000000
True
True
True True


### 3.4. Đếm số bit `1` liên tiếp ở đầu byte

Số bit `1` liên tiếp từ bit cao nhất của byte đầu chính là **số byte** của ký tự: `0xxxxxxx` (0 bit 1) là ký tự 1 byte; `110xxxxx` (2 bit 1) là 2 byte; `1110xxxx` (3 bit 1) là 3 byte; `11110xxx` (4 bit 1) là 4 byte. Còn `10xxxxxx` (1 bit 1) chỉ được xuất hiện làm byte tiếp theo; `11111xxx` (5 bit 1 trở lên) không hợp lệ.

In [5]:
def leading_ones(v):
    """Đếm số bit 1 liên tiếp tính từ bit cao nhất của byte 8 bit."""
    count = 0
    for shift in range(7, -1, -1):
        if (v >> shift) & 1:
            count += 1
        else:
            break
    return count


for v in [1, 0b10000010, 197, 235, 0b11110000, 0b11111000, 255]:
    print(f"{format(v, '08b')}  leading_ones = {leading_ones(v)}")

00000001  leading_ones = 0
10000010  leading_ones = 1
11000101  leading_ones = 2
11101011  leading_ones = 3
11110000  leading_ones = 4
11111000  leading_ones = 5
11111111  leading_ones = 8


### 3.5. Duyệt mảng với biến trạng thái `cnt`

Khi duyệt `for v in data`, ta cần *nhớ* điều gì đó giữa các phần tử: "ký tự hiện tại còn đợi bao nhiêu byte tiếp theo". Một biến nguyên `cnt` đủ làm việc này. Nếu `cnt > 0` thì byte hiện tại **phải** là byte tiếp theo (`10xxxxxx`) và ta giảm `cnt`; nếu `cnt == 0` thì byte hiện tại **bắt đầu** một ký tự mới. Kết thúc duyệt mà `cnt > 0` nghĩa là ký tự cuối bị cụt.

In [6]:
# ví dụ nhỏ về "trạng thái giữa các phần tử": đếm số lần ngắt khi tổng chạm 10
cnt = 0
for x in [3, 4, 5, 6]:
    cnt += x
    print("x =", x, " cnt =", cnt)

x = 3  cnt = 3
x = 4  cnt = 7
x = 5  cnt = 12
x = 6  cnt = 18


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Quét ví dụ 1 `[197, 130, 1]` từ trái sang phải như một người đọc:

| Byte | Nhị phân | Nhìn thấy gì | Kết luận |
|---|---|---|---|
| 197 | `11000101` | bắt đầu `110` | ký tự **2 byte**: cần **1** byte `10xxxxxx` nữa |
| 130 | `10000010` | bắt đầu `10` | đúng là byte tiếp theo, hết ký tự |
| 1 | `00000001` | bắt đầu `0` | ký tự **1 byte**, xong |

Hết dãy và không còn byte nào đang chờ ⇒ **hợp lệ**.

Ví dụ 2 `[235, 140, 4]`: `235 = 11101011` (bắt đầu `1110`) ⇒ cần **2** byte `10xxxxxx`. `140 = 10001100` ✓ (còn cần 1). `4 = 00000100` bắt đầu `00` ≠ `10` ✗ ⇒ **không hợp lệ**.

Quy tắc rút ra: *mỗi byte hoặc là bắt đầu một ký tự mới (khi không còn gì phải chờ), hoặc là byte tiếp theo (khi đang chờ)*; ta chỉ cần biết còn chờ bao nhiêu byte.

## 5. Cách 1 — Brute Force: đổi sang chuỗi bit rồi kiểm tra bằng `startswith`

### Ý tưởng

1. Đổi mỗi số thành chuỗi 8 ký tự `0/1`: `bits = [format(b, '08b') for b in data]`.
2. Dùng chỉ số `i`. Nhìn chuỗi `bits[i]`: nếu bắt đầu bằng `0` ⇒ ký tự dài `k = 1`; `110` ⇒ `k = 2`; `1110` ⇒ `k = 3`; `11110` ⇒ `k = 4`; còn lại (`10...`, `11111...`) ⇒ sai.
3. Nếu `i + k > n` (thiếu byte) ⇒ sai. Kiểm tra `bits[i+1 .. i+k-1]` đều `startswith('10')`.
4. `i += k` và lặp.

In [7]:
def valid_utf8_strings(data):
    bits = [format(b, '08b') for b in data]
    n = len(bits)
    i = 0
    while i < n:
        s = bits[i]
        if s.startswith('0'):
            k = 1
        elif s.startswith('110'):
            k = 2
        elif s.startswith('1110'):
            k = 3
        elif s.startswith('11110'):
            k = 4
        else:
            return False                  # 10xxxxxx hoặc 11111xxx không được làm byte đầu
        if i + k > n:
            return False                  # thiếu byte
        for j in range(i + 1, i + k):
            if not bits[j].startswith('10'):
                return False
        i += k
    return True


for data, expected in examples:
    print(data, valid_utf8_strings(data), expected)
print(valid_utf8_strings([197]), valid_utf8_strings([0]), valid_utf8_strings([255]))

[197, 130, 1] True True
[235, 140, 4] False False
False True False


### Độ phức tạp & nút thắt

- Mỗi byte được đổi sang chuỗi (hằng số 8 ký tự) và xét một lần ⇒ **Time `O(n)`**, **Space `O(n)`** (mảng `bits`).
- Đúng, nhưng **cồng kềnh**: tạo `n` chuỗi nhỏ chỉ để nhìn vài bit đầu; code dài với nhiều `startswith`.

**Nút thắt:** ta chỉ cần vài bit đầu của mỗi byte — dịch bit làm việc đó trực tiếp trên số nguyên, không tốn bộ nhớ phụ.

## 6. Ý tưởng thuật toán / bước đột phá

Bỏ mảng `bits`. Dùng **một biến đếm `cnt`** = số byte tiếp theo (dạng `10xxxxxx`) còn phải xuất hiện:

- Nếu `cnt > 0`: byte hiện tại **phải** có 2 bit cao là `10` (`v >> 6 == 0b10`); đúng thì `cnt -= 1`, sai thì trả `False`.
- Nếu `cnt == 0`: byte hiện tại **mở đầu** một ký tự:
  - `v >> 7 == 0` (`0xxxxxxx`) ⇒ ký tự 1 byte, `cnt = 0`;
  - `v >> 5 == 0b110` ⇒ `cnt = 1`;
  - `v >> 4 == 0b1110` ⇒ `cnt = 2`;
  - `v >> 3 == 0b11110` ⇒ `cnt = 3`;
  - còn lại ⇒ `False`.
- Sau khi duyệt hết: hợp lệ **khi và chỉ khi `cnt == 0`** (không còn ký tự cụt).

Thứ tự kiểm tra quan trọng: **phải kiểm tra `cnt > 0` trước**, vì byte `10xxxxxx` hợp lệ khi đang chờ nhưng không hợp lệ khi mở đầu.

In [8]:
# Bảng "mở đầu ký tự" bằng dịch bit: v >> s so sánh với tiền tố
for v in [0b01111111, 0b11000101, 0b11101011, 0b11110111, 0b11111000, 0b10000010]:
    print(format(v, '08b'),
          "1B" if v >> 7 == 0 else
          "2B" if v >> 5 == 0b110 else
          "3B" if v >> 4 == 0b1110 else
          "4B" if v >> 3 == 0b11110 else "KHÔNG hợp lệ làm byte đầu")

01111111 1B
11000101 2B
11101011 3B
11110111 4B
11111000 KHÔNG hợp lệ làm byte đầu
10000010 KHÔNG hợp lệ làm byte đầu


## 7. Các cách tiếp theo

Tổng cộng **5 cách**: (1) chuỗi bit + `startswith`; (2) biến đếm + dịch bit (cách chuẩn); (3) nhảy theo từng ký tự; (4) bảng tra 256 phần tử; (5) biểu thức chính quy trên chuỗi bit.

### Cách 2 — Một lượt, biến đếm `cnt` + dịch bit

Giải thích từng đoạn:

- `cnt = 0`.
- Với mỗi `v`: nếu `cnt > 0` kiểm tra tiền tố `10` rồi `cnt -= 1`.
- Ngược lại phân loại byte đầu bằng chuỗi `elif` như ở mục 6.
- `return cnt == 0`.

In [9]:
def valid_utf8_counter(data):
    cnt = 0
    for v in data:
        if cnt > 0:
            if v >> 6 != 0b10:
                return False
            cnt -= 1
        elif v >> 7 == 0:
            cnt = 0
        elif v >> 5 == 0b110:
            cnt = 1
        elif v >> 4 == 0b1110:
            cnt = 2
        elif v >> 3 == 0b11110:
            cnt = 3
        else:
            return False
    return cnt == 0


for data, expected in examples:
    print(data, valid_utf8_counter(data), expected)

[197, 130, 1] True True
[235, 140, 4] False False


**Code nộp LeetCode (Cách 2):**

In [10]:
class SolutionCounter:
    def validUtf8(self, data: list[int]) -> bool:
        cnt = 0
        for v in data:
            if cnt > 0:
                if v >> 6 != 0b10:
                    return False
                cnt -= 1
            elif v >> 7 == 0:
                cnt = 0
            elif v >> 5 == 0b110:
                cnt = 1
            elif v >> 4 == 0b1110:
                cnt = 2
            elif v >> 3 == 0b11110:
                cnt = 3
            else:
                return False
        return cnt == 0


print([SolutionCounter().validUtf8(d) for d, _ in examples])

[True, False]


Độ phức tạp: **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — Nhảy theo từng ký tự (con trỏ `i`)

Thay vì đếm lùi, ta **xác định độ dài ký tự `k`** từ byte đầu (qua hàm `char_len(v)` trả `1..4` hoặc `0` nếu không hợp lệ), kiểm tra còn đủ `k − 1` byte phía sau và tất cả đều có dạng `10xxxxxx` (dùng `all(...)` trên lát cắt), rồi **nhảy** `i += k`.

Khác Cách 2 ở chỗ: xử lý theo "khối ký tự" thay vì từng byte.

In [11]:
def char_len(v):
    if v >> 7 == 0:
        return 1
    if v >> 5 == 0b110:
        return 2
    if v >> 4 == 0b1110:
        return 3
    if v >> 3 == 0b11110:
        return 4
    return 0                          # không hợp lệ làm byte đầu


def valid_utf8_jump(data):
    n = len(data)
    i = 0
    while i < n:
        k = char_len(data[i])
        if k == 0 or i + k > n:
            return False
        if not all(b >> 6 == 0b10 for b in data[i + 1:i + k]):
            return False
        i += k
    return True


for data, expected in examples:
    print(data, valid_utf8_jump(data), expected)

[197, 130, 1] True True
[235, 140, 4] False False


**Code nộp LeetCode (Cách 3):**

In [12]:
class SolutionJump:
    def validUtf8(self, data: list[int]) -> bool:
        def char_len(v):
            if v >> 7 == 0:
                return 1
            if v >> 5 == 0b110:
                return 2
            if v >> 4 == 0b1110:
                return 3
            if v >> 3 == 0b11110:
                return 4
            return 0

        n, i = len(data), 0
        while i < n:
            k = char_len(data[i])
            if k == 0 or i + k > n:
                return False
            if not all(b >> 6 == 0b10 for b in data[i + 1:i + k]):
                return False
            i += k
        return True


print([SolutionJump().validUtf8(d) for d, _ in examples])

[True, False]


Độ phức tạp: **Time `O(n)`** (mỗi byte thuộc đúng một ký tự, được xét một lần), **Space `O(1)`** (lát cắt `data[i+1:i+k]` tối đa 3 phần tử).

### Cách 4 — Bảng tra 256 phần tử (state machine)

Chỉ có **256 giá trị byte** khác nhau. Ta tính trước, cho mỗi giá trị, **loại** của nó:

- `'C'` (continuation) nếu dạng `10xxxxxx`;
- số `0, 1, 2, 3` = số byte tiếp theo cần có nếu nó là byte đầu hợp lệ (`0xxxxxxx` → 0, `110...` → 1, `1110...` → 2, `11110...` → 3);
- `-1` nếu không hợp lệ làm byte đầu (`11111xxx`).

Sau đó mỗi lần xử lý chỉ cần **tra bảng** thay vì chuỗi `elif`. Cùng một máy trạng thái có `need` (số byte tiếp theo cần) làm trạng thái.

In [13]:
CONT = -2          # ký hiệu: byte tiếp theo 10xxxxxx
BAD = -1           # ký hiệu: không hợp lệ làm byte đầu


def build_table():
    table = []
    for v in range(256):
        if v >> 6 == 0b10:
            table.append(CONT)
        elif v >> 7 == 0:
            table.append(0)
        elif v >> 5 == 0b110:
            table.append(1)
        elif v >> 4 == 0b1110:
            table.append(2)
        elif v >> 3 == 0b11110:
            table.append(3)
        else:
            table.append(BAD)
    return table


TABLE = build_table()
print(TABLE[0], TABLE[127], TABLE[128], TABLE[191], TABLE[192], TABLE[224], TABLE[240], TABLE[248], TABLE[255])
from collections import Counter
print(Counter(TABLE))               # 128 giá trị 0, 64 CONT, 32 giá trị 1, 16 giá trị 2, 8 giá trị 3, 8 BAD


def valid_utf8_table(data):
    need = 0
    for v in data:
        t = TABLE[v & 0xFF]
        if need > 0:
            if t != CONT:
                return False
            need -= 1
        else:
            if t < 0:                 # CONT hoặc BAD làm byte đầu
                return False
            need = t
    return need == 0


for data, expected in examples:
    print(data, valid_utf8_table(data), expected)

0 0 -2 -2 1 2 3 -1 -1
Counter({0: 128, -2: 64, 1: 32, 2: 16, 3: 8, -1: 8})
[197, 130, 1] True True
[235, 140, 4] False False


**Code nộp LeetCode (Cách 4):**

In [14]:
class SolutionTable:
    def validUtf8(self, data: list[int]) -> bool:
        table = []
        for v in range(256):
            if v >> 6 == 0b10:
                table.append(-2)
            elif v >> 7 == 0:
                table.append(0)
            elif v >> 5 == 0b110:
                table.append(1)
            elif v >> 4 == 0b1110:
                table.append(2)
            elif v >> 3 == 0b11110:
                table.append(3)
            else:
                table.append(-1)
        need = 0
        for v in data:
            t = table[v & 0xFF]
            if need > 0:
                if t != -2:
                    return False
                need -= 1
            else:
                if t < 0:
                    return False
                need = t
        return need == 0


print([SolutionTable().validUtf8(d) for d, _ in examples])

[True, False]


Độ phức tạp: **Time `O(n)`** (+ `O(256)` dựng bảng), **Space `O(256) = O(1)`**. Đáng dùng khi phải kiểm tra rất nhiều dữ liệu (bảng dựng một lần).

### Cách 5 — Biểu thức chính quy trên chuỗi bit

Nối tất cả byte thành **một chuỗi bit dài** rồi kiểm tra bằng `re.fullmatch` với mẫu mô tả đúng bảng ở mục 2:

```
(?:0[01]{7} | 110[01]{5}10[01]{6} | 1110[01]{4}(?:10[01]{6}){2} | 11110[01]{3}(?:10[01]{6}){3})*
```

`*` ở cuối: lặp lại 0 hoặc nhiều ký tự hợp lệ. `fullmatch` đòi **cả chuỗi** khớp. Cách này nhìn gần như là sao chép đề bài, và độc lập hoàn toàn với dịch bit ⇒ ta sẽ dùng nó làm **oracle** kiểm chứng ở mục 12. (Vì các nhánh `0`, `110`, `1110`, `11110` loại trừ nhau theo tiền tố nên regex không bị backtracking nặng.)

In [15]:
import re

UTF8_RE = re.compile(
    r'(?:0[01]{7}'
    r'|110[01]{5}10[01]{6}'
    r'|1110[01]{4}(?:10[01]{6}){2}'
    r'|11110[01]{3}(?:10[01]{6}){3})*'
)


def valid_utf8_regex(data):
    bits = ''.join(format(b, '08b') for b in data)
    return UTF8_RE.fullmatch(bits) is not None


for data, expected in examples:
    print(data, valid_utf8_regex(data), expected)
print(valid_utf8_regex([197]), valid_utf8_regex([0, 0]), valid_utf8_regex([255]))

[197, 130, 1] True True
[235, 140, 4] False False
False True False


**Code nộp LeetCode (Cách 5):**

In [16]:
import re


class SolutionRegex:
    PATTERN = re.compile(
        r'(?:0[01]{7}'
        r'|110[01]{5}10[01]{6}'
        r'|1110[01]{4}(?:10[01]{6}){2}'
        r'|11110[01]{3}(?:10[01]{6}){3})*'
    )

    def validUtf8(self, data: list[int]) -> bool:
        bits = ''.join(format(b, '08b') for b in data)
        return self.PATTERN.fullmatch(bits) is not None


print([SolutionRegex().validUtf8(d) for d, _ in examples])

[True, False]


Độ phức tạp: **Time `O(n)`** (chuỗi bit dài `8n`, khớp tuyến tính), **Space `O(n)`** cho chuỗi bit. Là cách *thú vị* nhưng tốn bộ nhớ hơn; trong phỏng vấn nên dùng Cách 2.

## 8. TRACE từng bước

### Bảng trạng thái cho Cách 2 trên `[235, 140, 4]`

| Bước | `v` | Nhị phân | `cnt` trước | Nhánh | `cnt` sau |
|---|---|---|---|---|---|
| 1 | 235 | `11101011` | 0 | `v >> 4 == 0b1110` ⇒ ký tự 3 byte | 2 |
| 2 | 140 | `10001100` | 2 | `cnt > 0`: `v >> 6 == 0b10` ✓ | 1 |
| 3 | 4 | `00000100` | 1 | `cnt > 0`: `v >> 6 = 0b00 ≠ 0b10` ✗ | **trả `False`** |

Và trên `[197, 130, 1]`:

| Bước | `v` | Nhị phân | `cnt` trước | Nhánh | `cnt` sau |
|---|---|---|---|---|---|
| 1 | 197 | `11000101` | 0 | `v >> 5 == 0b110` ⇒ ký tự 2 byte | 1 |
| 2 | 130 | `10000010` | 1 | tiền tố `10` ✓ | 0 |
| 3 | 1 | `00000001` | 0 | `v >> 7 == 0` ⇒ ký tự 1 byte | 0 |

Hết mảng với `cnt == 0` ⇒ `True`.

In [17]:
def trace_counter(data):
    cnt = 0
    for step, v in enumerate(data, 1):
        b = format(v, '08b')
        before = cnt
        if cnt > 0:
            ok = v >> 6 == 0b10
            print(f"bước {step}: v={v:3} {b} cnt={before}>0 -> cần tiền tố 10, thấy {b[:2]} -> {'OK' if ok else 'SAI'}")
            if not ok:
                print("=> False")
                return False
            cnt -= 1
        elif v >> 7 == 0:
            print(f"bước {step}: v={v:3} {b} cnt=0 -> ký tự 1 byte")
        elif v >> 5 == 0b110:
            cnt = 1
            print(f"bước {step}: v={v:3} {b} cnt=0 -> mở ký tự 2 byte, cnt=1")
        elif v >> 4 == 0b1110:
            cnt = 2
            print(f"bước {step}: v={v:3} {b} cnt=0 -> mở ký tự 3 byte, cnt=2")
        elif v >> 3 == 0b11110:
            cnt = 3
            print(f"bước {step}: v={v:3} {b} cnt=0 -> mở ký tự 4 byte, cnt=3")
        else:
            print(f"bước {step}: v={v:3} {b} cnt=0 -> KHÔNG hợp lệ làm byte đầu")
            print("=> False")
            return False
    print(f"hết mảng, cnt = {cnt} ->", cnt == 0)
    return cnt == 0


trace_counter([197, 130, 1])
print()
trace_counter([235, 140, 4])
print()
trace_counter([240, 162, 138])      # ký tự 4 byte bị cụt
print()
trace_counter([145])                # byte tiếp theo đứng một mình

bước 1: v=197 11000101 cnt=0 -> mở ký tự 2 byte, cnt=1
bước 2: v=130 10000010 cnt=1>0 -> cần tiền tố 10, thấy 10 -> OK
bước 3: v=  1 00000001 cnt=0 -> ký tự 1 byte
hết mảng, cnt = 0 -> True

bước 1: v=235 11101011 cnt=0 -> mở ký tự 3 byte, cnt=2
bước 2: v=140 10001100 cnt=2>0 -> cần tiền tố 10, thấy 10 -> OK
bước 3: v=  4 00000100 cnt=1>0 -> cần tiền tố 10, thấy 00 -> SAI
=> False

bước 1: v=240 11110000 cnt=0 -> mở ký tự 4 byte, cnt=3
bước 2: v=162 10100010 cnt=3>0 -> cần tiền tố 10, thấy 10 -> OK
bước 3: v=138 10001010 cnt=2>0 -> cần tiền tố 10, thấy 10 -> OK
hết mảng, cnt = 1 -> False

bước 1: v=145 10010001 cnt=0 -> KHÔNG hợp lệ làm byte đầu
=> False


False

### Trace cho Cách 3 (nhảy) trên `[197, 130, 1, 226, 130, 172]`

| `i` | `data[i]` | `k` = độ dài ký tự | các byte tiếp theo cần `10` | nhảy tới |
|---|---|---|---|---|
| 0 | 197 | 2 | `[130]` ✓ | 2 |
| 2 | 1 | 1 | (không có) | 3 |
| 3 | 226 | 3 | `[130, 172]` ✓ | 6 = `n` ⇒ hết |

In [18]:
def trace_jump(data):
    n, i = len(data), 0
    while i < n:
        k = char_len(data[i])
        follow = data[i + 1:i + k]
        print(f"i={i} data[i]={data[i]:3} ({format(data[i], '08b')}) k={k} theo sau={follow} "
              f"-> {[format(b >> 6, '02b') for b in follow]}")
        if k == 0 or i + k > n or not all(b >> 6 == 0b10 for b in follow):
            print("=> False")
            return False
        i += k
    print("=> True")
    return True


trace_jump([197, 130, 1, 226, 130, 172])
trace_jump([226, 130])

i=0 data[i]=197 (11000101) k=2 theo sau=[130] -> ['10']
i=2 data[i]=  1 (00000001) k=1 theo sau=[] -> []
i=3 data[i]=226 (11100010) k=3 theo sau=[130, 172] -> ['10', '10']
=> True
i=0 data[i]=226 (11100010) k=3 theo sau=[130] -> ['10']
=> False


False

## 9. Code nộp LeetCode

Chọn **Cách 2** làm bản nộp chính (một lượt, `O(1)` bộ nhớ, ngắn, dễ giải thích):

```python
class Solution:
    def validUtf8(self, data: List[int]) -> bool:
        cnt = 0
        for v in data:
            if cnt > 0:
                if v >> 6 != 0b10:
                    return False
                cnt -= 1
            elif v >> 7 == 0:
                cnt = 0
            elif v >> 5 == 0b110:
                cnt = 1
            elif v >> 4 == 0b1110:
                cnt = 2
            elif v >> 3 == 0b11110:
                cnt = 3
            else:
                return False
        return cnt == 0
```

(Trong notebook này dùng `list[int]` vì ta chạy ngoài môi trường LeetCode nên không có sẵn `List` từ `typing`.)

In [19]:
class Solution:
    def validUtf8(self, data: list[int]) -> bool:
        cnt = 0
        for v in data:
            if cnt > 0:
                if v >> 6 != 0b10:
                    return False
                cnt -= 1
            elif v >> 7 == 0:
                cnt = 0
            elif v >> 5 == 0b110:
                cnt = 1
            elif v >> 4 == 0b1110:
                cnt = 2
            elif v >> 3 == 0b11110:
                cnt = 3
            else:
                return False
        return cnt == 0


for data, expected in examples:
    got = Solution().validUtf8(data)
    print(data, got, expected, got == expected)

[197, 130, 1] True True True
[235, 140, 4] False False True


## 10. Vì sao đúng

**Bất biến:** sau khi xử lý xong `j` byte đầu của `data`, biến `cnt` bằng đúng **số byte tiếp theo mà ký tự đang dở (nếu có) còn thiếu**, và `cnt = 0` nếu `j` byte đó ghép thành các ký tự hoàn chỉnh; hơn nữa mọi byte đã xử lý đều đúng vai trò của nó (byte đầu hợp lệ hoặc byte tiếp theo `10xxxxxx`).

- **Khởi tạo:** `j = 0`, `cnt = 0` — chưa có gì.
- **Duy trì:**
  - Nếu `cnt > 0`, theo luật UTF-8 byte kế **bắt buộc** là `10xxxxxx` (đúng điều ta kiểm tra bằng `v >> 6 == 0b10`); đúng thì còn thiếu một byte ít hơn.
  - Nếu `cnt == 0`, byte kế phải **mở đầu** ký tự mới; bốn dạng tiền tố `0`, `110`, `1110`, `11110` ứng với 1–4 byte và loại trừ lẫn nhau; mọi dạng khác (`10xxxxxx` hay `11111xxx`) không thể là byte đầu ⇒ sai ngay.
- **Kết thúc:** nếu `cnt == 0` thì dãy tách được thành các ký tự trọn vẹn ⇒ hợp lệ; nếu `cnt > 0` thì ký tự cuối bị cụt ⇒ không hợp lệ.

Mọi lỗi định dạng đều bị phát hiện ngay ở byte đầu tiên vi phạm (trả `False` sớm), và dãy hợp lệ không bao giờ bị từ chối vì mọi nhánh trả `False` đều ứng với đúng một vi phạm của luật.

## 11. Time/Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Chuỗi bit + `startswith` | `O(n)` | `O(n)` |
| 2. Biến đếm + dịch bit | **`O(n)`** | **`O(1)`** |
| 3. Nhảy theo ký tự | `O(n)` | `O(1)` |
| 4. Bảng tra 256 | `O(n)` (+256) | `O(1)` |
| 5. Regex trên chuỗi bit | `O(n)` | `O(n)` |

`n = len(data)`. Mọi cách đều duyệt mỗi byte một số lần cố định.

## 12. Bộ test

### 12.1. Ví dụ đề + edge case

In [20]:
methods = [valid_utf8_strings, valid_utf8_counter, valid_utf8_jump, valid_utf8_table, valid_utf8_regex,
           SolutionCounter().validUtf8, SolutionJump().validUtf8, SolutionTable().validUtf8,
           SolutionRegex().validUtf8, Solution().validUtf8]

cases = [
    ([197, 130, 1], True),
    ([235, 140, 4], False),
    ([0], True),                              # 1 byte nhỏ nhất
    ([127], True),                            # 1 byte lớn nhất 01111111
    ([128], False),                           # 10000000 không được làm byte đầu
    ([255], False),                           # 11111111
    ([248, 130, 130, 130, 130], False),       # 11111000: 5 byte không tồn tại
    ([197], False),                           # ký tự 2 byte bị cụt
    ([197, 130], True),
    ([197, 130, 130], False),                 # thừa một byte tiếp theo đứng lẻ
    ([226, 130, 172], True),                  # 3 byte (ký tự €)
    ([226, 130], False),                      # 3 byte nhưng thiếu 1
    ([240, 144, 128, 128], True),             # 4 byte
    ([240, 144, 128], False),
    ([240, 144, 128, 127], False),            # byte cuối không phải 10xxxxxx
    ([230, 136, 145], True),
    ([145], False),                           # byte tiếp theo đứng một mình
    ([0, 197, 130, 0], True),
    ([197, 197, 130], False),                 # 197 cần 130 nhưng gặp 197 (110xxxxx)
    ([192, 128], True),                       # theo luật đề: 11000000 10000000 hợp lệ (dù thực tế "overlong")
    ([0] * 20000, True),
    ([197, 130] * 10000, True),
    ([197, 130] * 9999 + [197], False),
]
for data, expected in cases:
    for f in methods:
        assert f(data) == expected, (f, data[:6], expected)
print("OK:", len(cases), "test cố định x", len(methods), "cách")

OK: 23 test cố định x 10 cách


### 12.2. Phản ví dụ: `[192, 128]` là hợp lệ theo đề nhưng Python từ chối

Bộ giải mã UTF-8 thật của Python **chặt hơn** đề: nó cấm "overlong" (mã hóa một ký tự bằng nhiều byte hơn cần), cấm surrogate, cấm điểm mã > U+10FFFF. Đề LeetCode chỉ kiểm tra **hình dạng** bit như bảng ở mục 2, nên không thể dùng `bytes(data).decode('utf-8')` làm đáp án. Có thể dùng nó như một **điều kiện đủ**: nếu Python giải mã được thì đề chắc chắn trả `True`.

In [21]:
def python_decodes(data):
    try:
        bytes(data).decode('utf-8')
        return True
    except UnicodeDecodeError:
        return False


print(python_decodes([192, 128]), Solution().validUtf8([192, 128]))      # False True  (khác nhau!)
print(python_decodes([197, 130, 1]), Solution().validUtf8([197, 130, 1]))
print(python_decodes([237, 160, 128]), Solution().validUtf8([237, 160, 128]))   # surrogate: Python False, đề True

False True
True True
False True


### 12.3. Dữ liệu thật: sinh từ `str.encode('utf-8')` (oracle độc lập cho trường hợp `True`)

Ta sinh các ký tự Unicode ngẫu nhiên (bỏ vùng surrogate), mã hóa bằng bộ mã hóa **chuẩn của Python**; kết quả chắc chắn là UTF-8 hợp lệ ⇒ mọi cách phải trả `True`.

In [22]:
import random

rng = random.Random(393)


def random_text(rng, length):
    chars = []
    for _ in range(length):
        kind = rng.choice([1, 2, 3, 4])
        if kind == 1:
            cp = rng.randint(0, 0x7F)
        elif kind == 2:
            cp = rng.randint(0x80, 0x7FF)
        elif kind == 3:
            cp = rng.randint(0x800, 0xFFFF)
            if 0xD800 <= cp <= 0xDFFF:
                cp = 0x20AC
        else:
            cp = rng.randint(0x10000, 0x10FFFF)
        chars.append(chr(cp))
    return ''.join(chars)


for _ in range(500):
    text = random_text(rng, rng.randint(1, 8))
    data = list(text.encode('utf-8'))
    for f in methods:
        assert f(data) is True, (f, data)
print("OK: 500 chuỗi UTF-8 thật đều hợp lệ")

OK: 500 chuỗi UTF-8 thật đều hợp lệ


### 12.4. Ngẫu nhiên đối chiếu 5 cách với regex (oracle độc lập)

Tạo dữ liệu từ nhiều nguồn: (a) UTF-8 thật rồi **đột biến** (đổi 1 byte, cắt cụt, chèn byte rác), (b) mảng byte hoàn toàn ngẫu nhiên, (c) chuỗi các byte "gần hợp lệ" (chỉ chọn từ các byte đầu/tiếp theo). Chỉ **regex** (mô tả đúng bảng của đề) làm chuẩn.

In [23]:
rng = random.Random(3930)
specials = [0, 1, 127, 128, 191, 192, 193, 223, 224, 239, 240, 247, 248, 255]


def mutate(data, rng):
    data = list(data)
    op = rng.randint(0, 3)
    if op == 0 and data:
        data[rng.randrange(len(data))] = rng.choice(specials + [rng.randint(0, 255)])
    elif op == 1 and len(data) > 1:
        data = data[:rng.randint(1, len(data) - 1)]
    elif op == 2:
        data.insert(rng.randint(0, len(data)), rng.choice(specials))
    return data or [0]


total_true = total_false = 0
for it in range(6000):
    mode = rng.randint(0, 2)
    if mode == 0:
        data = mutate(list(random_text(rng, rng.randint(1, 6)).encode('utf-8')), rng)
    elif mode == 1:
        data = [rng.randint(0, 255) for _ in range(rng.randint(1, 8))]
    else:
        data = [rng.choice(specials) for _ in range(rng.randint(1, 6))]
    expected = valid_utf8_regex(data)
    total_true += expected
    total_false += not expected
    for f in methods:
        assert f(data) == expected, (f, data, expected)
    # nếu Python giải mã được thì đề cũng phải True
    if python_decodes(data):
        assert expected
print("OK: 6000 input ngẫu nhiên, 10 hàm khớp nhau; số True:", total_true, "số False:", total_false)

OK: 6000 input ngẫu nhiên, 10 hàm khớp nhau; số True: 1193 số False: 4807


### 12.5. Vét cạn mọi mảng ngắn từ tập byte đặc trưng

Với tập 8 giá trị đại diện cho mọi "loại" byte: `0, 127, 128, 191, 192, 224, 240, 248` và độ dài tới 4, ta duyệt **tất cả** `8^1 + ... + 8^4` mảng.

In [24]:
from itertools import product

reps = [0, 127, 128, 191, 192, 224, 240, 248]
count = 0
for length in range(1, 5):
    for tup in product(reps, repeat=length):
        data = list(tup)
        expected = valid_utf8_regex(data)
        assert Solution().validUtf8(data) == expected, data
        assert valid_utf8_table(data) == expected and valid_utf8_jump(data) == expected, data
        count += 1
print("OK: vét cạn", count, "mảng")

OK: vét cạn 4680 mảng


### 12.6. Hiệu năng (n = 2·10^4)

In [25]:
import time

big_ok = ([226, 130, 172] * 6666) + [65, 66]            # ~2*10^4 byte, hợp lệ
big_bad = ([226, 130, 172] * 6666) + [65, 130]            # byte cuối là continuation lẻ
for name, f in [("counter", Solution().validUtf8), ("jump", SolutionJump().validUtf8),
                ("table", SolutionTable().validUtf8), ("regex", SolutionRegex().validUtf8),
                ("strings", valid_utf8_strings)]:
    t0 = time.perf_counter()
    a, b = f(big_ok), f(big_bad)
    print(f"{name:8} {a} {b}  {time.perf_counter() - t0:.4f}s")
    assert a is True and b is False

counter  True False  0.0028s
jump     True False  0.0086s
table    True False  0.0026s
regex    True False  0.0156s
strings  True False  0.0153s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên kiểm tra `cnt == 0` ở cuối

In [26]:
def bug_no_final_check(data):
    cnt = 0
    for v in data:
        if cnt > 0:
            if v >> 6 != 0b10:
                return False
            cnt -= 1
        elif v >> 7 == 0:
            cnt = 0
        elif v >> 5 == 0b110:
            cnt = 1
        elif v >> 4 == 0b1110:
            cnt = 2
        elif v >> 3 == 0b11110:
            cnt = 3
        else:
            return False
    return True                              # SAI: thiếu "return cnt == 0"


print(bug_no_final_check([197]), "(đúng là", Solution().validUtf8([197]), ")")
print(bug_no_final_check([226, 130]), "(đúng là", Solution().validUtf8([226, 130]), ")")

True (đúng là False )
True (đúng là False )


**Vì sao sai:** nếu mảng kết thúc khi ký tự còn đang dở (`cnt > 0`), dữ liệu bị cụt nhưng không có byte nào vi phạm để phát hiện trong vòng lặp. **Sửa:** `return cnt == 0`.

### Lỗi 2: chỉ đếm, không kiểm tra byte tiếp theo có tiền tố `10`

In [27]:
def bug_no_prefix_check(data):
    cnt = 0
    for v in data:
        if cnt > 0:
            cnt -= 1                          # SAI: không kiểm tra v >> 6 == 0b10
        elif v >> 7 == 0:
            cnt = 0
        elif v >> 5 == 0b110:
            cnt = 1
        elif v >> 4 == 0b1110:
            cnt = 2
        elif v >> 3 == 0b11110:
            cnt = 3
        else:
            return False
    return cnt == 0


print(bug_no_prefix_check([235, 140, 4]), "(đúng là", Solution().validUtf8([235, 140, 4]), ")")

True (đúng là False )


**Vì sao sai:** đúng đề ví dụ 2: số lượng byte vừa đủ nhưng byte cuối `00000100` không có dạng `10xxxxxx`. **Sửa:** khi `cnt > 0` phải xác nhận `v >> 6 == 0b10`.

### Lỗi 3: nhận mọi byte có 2 bit đầu `11` làm ký tự 2 byte

In [28]:
def bug_two_bits(data):
    cnt = 0
    for v in data:
        if cnt > 0:
            if v >> 6 != 0b10:
                return False
            cnt -= 1
        elif v >> 7 == 0:
            cnt = 0
        elif v >> 6 == 0b11:                  # SAI: gộp 110, 1110, 11110, 11111 làm một
            cnt = 1
        else:
            return False
    return cnt == 0


print(bug_two_bits([226, 130, 172]), "(đúng là", Solution().validUtf8([226, 130, 172]), ")")   # ký tự €
print(bug_two_bits([255, 130]), "(đúng là", Solution().validUtf8([255, 130]), ")")

False (đúng là True )
True (đúng là False )


**Vì sao sai:** byte đầu `1110xxxx` yêu cầu **2** byte tiếp theo, `11110xxx` yêu cầu **3**; còn `11111xxx` không hợp lệ. Gộp chung làm cnt luôn `1`, nên `[226, 130, 172]` bị từ chối (byte thứ 3 trở thành "byte đầu" `10101100`) và `[255, 130]` lại bị nhận nhầm. **Sửa:** phân biệt đủ 4 dạng tiền tố với số bit dịch tương ứng (`>> 5`, `>> 4`, `>> 3`).

### Lỗi 4: cho phép byte `10xxxxxx` làm byte đầu

In [29]:
def bug_continuation_as_start(data):
    cnt = 0
    for v in data:
        if cnt > 0:
            if v >> 6 != 0b10:
                return False
            cnt -= 1
        elif v >> 5 == 0b110:
            cnt = 1
        elif v >> 4 == 0b1110:
            cnt = 2
        elif v >> 3 == 0b11110:
            cnt = 3
        elif v >> 6 == 0b10 or v >> 7 == 0:   # SAI: 10xxxxxx bị coi như ký tự 1 byte
            cnt = 0
        else:
            return False
    return cnt == 0


print(bug_continuation_as_start([145]), "(đúng là", Solution().validUtf8([145]), ")")
print(bug_continuation_as_start([197, 130, 130]), "(đúng là", Solution().validUtf8([197, 130, 130]), ")")

True (đúng là False )
True (đúng là False )


**Vì sao sai:** byte `10xxxxxx` chỉ hợp lệ khi đang là byte **tiếp theo** của một ký tự. Đứng một mình hay thừa ra đều sai. **Sửa:** khi `cnt == 0`, byte có dạng `10xxxxxx` rơi vào nhánh `else` ⇒ `False`.

## 14. Mẹo & nhận dạng pattern

- Đề cho **mảng byte/số nguyên** và nói về **"tiền tố bit"** ⇒ dùng `>>` (hoặc `&`) để lấy bit cao, so với hằng `0b...`.
- Bài kiểm tra **định dạng tuần tự** (mỗi phần tử phụ thuộc phần tử trước) ⇒ **máy trạng thái với một biến đếm**; nhớ đoạn kết `return cnt == 0`.
- Luôn xử lý **"đang chờ"** trước **"mở đầu mới"** (kiểm tra `cnt > 0` trước), vì cùng một byte có nghĩa khác nhau tùy trạng thái.
- Viết hằng nhị phân `0b110`, `0b10`... thay vì số thập phân giúp code tự giải thích.
- Dùng **regex trên chuỗi bit** như một oracle độc lập là kỹ thuật kiểm thử đáng nhớ; dùng **`bytes.decode`** chỉ như điều kiện đủ vì Python chặt hơn đề.
- Họ hàng: *Validate IP Address*, *Number of 1 Bits*, *Counting Bits*, *Binary Watch*, các bài parsing bằng state machine.

## 15. Biến thể

1. **Đếm số ký tự** trong dữ liệu hợp lệ (hoặc trả `-1` nếu sai): cộng thêm 1 mỗi khi mở đầu ký tự mới.
2. **Giải mã thành điểm mã Unicode** (code point): lấy các bit "payload" `x` của mỗi byte rồi ghép. Với hợp lệ chuẩn, có thể so với `bytes(data).decode('utf-8')`.
3. **UTF-8 chặt như chuẩn thật:** cấm overlong, cấm surrogate (`U+D800..U+DFFF`), cấm `> U+10FFFF` — khi đó kết quả phải trùng với việc `decode` của Python có thành công hay không.
4. **UTF-16 / UTF-32 validation** với quy tắc surrogate.
5. **Tìm vị trí lỗi đầu tiên** thay vì chỉ trả `True/False`.

Minh họa biến thể 1 và 2:

In [30]:
def count_chars(data):
    """Số ký tự nếu hợp lệ, ngược lại -1."""
    if not Solution().validUtf8(data):
        return -1
    return sum(1 for v in data if v >> 6 != 0b10)       # mỗi ký tự có đúng một byte "không phải continuation"


def decode_codepoints(data):
    """Giả sử data hợp lệ theo đề; ghép các bit payload thành điểm mã."""
    result, i = [], 0
    while i < len(data):
        k = char_len(data[i])
        cp = data[i] & (0x7F >> k) if k > 1 else data[i]     # bỏ phần tiền tố ở byte đầu
        for b in data[i + 1:i + k]:
            cp = (cp << 6) | (b & 0x3F)
        result.append(cp)
        i += k
    return result


text = "aé€𝄞"
data = list(text.encode('utf-8'))
print(data, count_chars(data), len(text))
print(decode_codepoints(data), [ord(c) for c in text])
assert count_chars(data) == len(text) and decode_codepoints(data) == [ord(c) for c in text]
assert count_chars([197]) == -1

# kiểm tra ngẫu nhiên với dữ liệu thật
rng = random.Random(1)
for _ in range(300):
    t = random_text(rng, rng.randint(1, 6))
    d = list(t.encode('utf-8'))
    assert count_chars(d) == len(t)
    assert decode_codepoints(d) == [ord(c) for c in t]
print("OK biến thể")

[97, 195, 169, 226, 130, 172, 240, 157, 132, 158] 4 4
[97, 233, 8364, 119070] [97, 233, 8364, 119070]
OK biến thể


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chuỗi bit + `startswith` | Đổi từng byte sang `'01...'` | `O(n)` | `O(n)` | Mới học, dễ nhìn |
| 2. Biến đếm + dịch bit | `cnt` = số byte tiếp theo còn thiếu | **`O(n)`** | **`O(1)`** | **Cách nộp chuẩn** |
| 3. Nhảy theo ký tự | Biết độ dài `k` rồi `all(...)` và `i += k` | `O(n)` | `O(1)` | Muốn tách thành các "khối ký tự" |
| 4. Bảng tra 256 | Phân loại byte bằng bảng dựng sẵn | `O(n)` | `O(1)` | Dữ liệu rất lớn, tránh nhiều `elif` |
| 5. Regex | Mẫu theo bảng đề trên chuỗi bit | `O(n)` | `O(n)` | **Oracle kiểm thử**, code ngắn |

## 17. Bài tập vận dụng (dễ → khó)

### Bài 1 (đoán output)

Đoán kết quả `bits_demo()` trước khi chạy.

In [31]:
def bits_demo():
    v = 197
    return format(v, '08b'), v >> 6, v >> 5, v >> 7


# Đoán: bits_demo() trả về tuple gì?
answer_1 = None   # TODO: điền một tuple 4 phần tử

### Bài 2 (tính tay)

Với `data = [226, 130, 172, 240, 144, 128]`, hỏi `validUtf8(data)` là `True` hay `False`? Chuyển từng byte sang nhị phân bằng tay và lập luận. Điền `answer_2`.

In [32]:
answer_2 = None   # TODO: True hoặc False

### Bài 3 (điền TODO)

Hoàn thiện Cách 2.

In [33]:
def valid_utf8_todo(data):
    cnt = 0
    for v in data:
        if cnt > 0:
            # TODO 1: nếu v không có tiền tố 10 -> return False; ngược lại giảm cnt
            ...
        elif v >> 7 == 0:
            cnt = 0
        # TODO 2: thêm các nhánh cho 110, 1110, 11110
        else:
            return False
    # TODO 3: trả kết quả cuối
    ...

### Bài 4 (tự viết lại)

Viết hàm `leading_ones_fast(v)` trả về số bit `1` liên tiếp ở đầu byte 8 bit **không dùng vòng `for` qua 8 vị trí**, mà dùng chuỗi: `format(v, '08b')` rồi đếm (gợi ý: `len(s) - len(s.lstrip('1'))`). Kiểm tra với `leading_ones` ở mục 3.4.

In [34]:
def leading_ones_fast(v):
    # TODO: tự viết
    pass

### Bài 5 (biến thể / mini challenge)

Viết `first_error_index(data)` trả về **chỉ số của byte đầu tiên gây lỗi** (hoặc `-1` nếu hợp lệ). Quy ước: nếu ký tự cuối bị cụt, trả về chỉ số của **byte đầu** của ký tự cụt đó. Ví dụ `first_error_index([197, 130, 1])` là `-1`; `first_error_index([235, 140, 4])` là `2`; `first_error_index([1, 197])` là `1`.

In [35]:
def first_error_index(data):
    # TODO: tự viết
    pass

## 18. Gợi ý (3 mức)

**Bài 1**
- Mức 1: `format(197, '08b')` là chuỗi 8 chữ số nhị phân của 197.
- Mức 2: `197 >> 6` bỏ 6 bit thấp, còn 2 bit cao; `>> 5` còn 3 bit; `>> 7` còn 1 bit.
- Mức 3: kết quả có dạng `('11000101', 3, 6, 1)` (số thập phân của `0b11`, `0b110`, `0b1`).

**Bài 2**
- Mức 1: `226 = 11100010` (3 byte) rồi hai byte `10...`.
- Mức 2: `240 = 11110000` mở ký tự 4 byte, cần 3 byte tiếp theo.
- Mức 3: sau `240` chỉ còn `144, 128` (2 byte) ⇒ thiếu 1 byte.

**Bài 3**
- Mức 1: `if v >> 6 != 0b10: return False` rồi `cnt -= 1`.
- Mức 2: các nhánh `elif v >> 5 == 0b110: cnt = 1` (rồi `1110 → 2`, `11110 → 3`).
- Mức 3: `return cnt == 0`.

**Bài 4**
- Mức 1: `s = format(v, '08b')`.
- Mức 2: `s.lstrip('1')` bỏ các chữ số `1` ở đầu.
- Mức 3: `return len(s) - len(s.lstrip('1'))`.

**Bài 5**
- Mức 1: giống Cách 3 (nhảy theo ký tự) nhưng trả chỉ số.
- Mức 2: ở mỗi `i`, tính `k = char_len(data[i])`; nếu `k == 0` ⇒ lỗi ở `i`. Nếu `i + k > n` ⇒ lỗi ở `i`. Duyệt `j` trong `i+1..i+k-1`: byte không phải `10xxxxxx` ⇒ lỗi ở `j`.
- Mức 3: không lỗi ⇒ `i += k`; hết vòng lặp ⇒ `-1`.

## 19. Đáp án tham khảo

In [36]:
# Bài 1
answer_1 = ('11000101', 3, 6, 1)
assert bits_demo() == answer_1

# Bài 2: 226,130,172 là một ký tự 3 byte; 240,144,128 thiếu 1 byte
answer_2 = False
assert answer_2 == Solution().validUtf8([226, 130, 172, 240, 144, 128])


# Bài 3
def valid_utf8_todo(data):
    cnt = 0
    for v in data:
        if cnt > 0:
            if v >> 6 != 0b10:
                return False
            cnt -= 1
        elif v >> 7 == 0:
            cnt = 0
        elif v >> 5 == 0b110:
            cnt = 1
        elif v >> 4 == 0b1110:
            cnt = 2
        elif v >> 3 == 0b11110:
            cnt = 3
        else:
            return False
    return cnt == 0


for data, expected in cases:
    assert valid_utf8_todo(data) == expected


# Bài 4
def leading_ones_fast(v):
    s = format(v, '08b')
    return len(s) - len(s.lstrip('1'))


assert all(leading_ones_fast(v) == leading_ones(v) for v in range(256))


# Bài 5
def first_error_index(data):
    n, i = len(data), 0
    while i < n:
        k = char_len(data[i])
        if k == 0 or i + k > n:
            return i
        for j in range(i + 1, i + k):
            if data[j] >> 6 != 0b10:
                return j
        i += k
    return -1


assert first_error_index([197, 130, 1]) == -1
assert first_error_index([235, 140, 4]) == 2
assert first_error_index([1, 197]) == 1
assert first_error_index([145]) == 0
# nhất quán với đáp án True/False
rng = random.Random(7)
for _ in range(2000):
    d = [rng.choice(specials) for _ in range(rng.randint(1, 6))]
    assert (first_error_index(d) == -1) == Solution().validUtf8(d)
print("Tất cả đáp án đã chạy đúng")

Tất cả đáp án đã chạy đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (biến đếm + dịch bit)

```python
cnt = 0                          # số byte "10xxxxxx" còn phải có
for v in data:
    if cnt > 0:                  # 1. đang chờ byte tiếp theo
        if v >> 6 != 0b10:
            return False
        cnt -= 1
    elif v >> 7 == 0:            # 2. ký tự 1 byte
        cnt = 0
    elif v >> 5 == 0b110:        #    ký tự 2 byte
        cnt = 1
    elif v >> 4 == 0b1110:       #    ký tự 3 byte
        cnt = 2
    elif v >> 3 == 0b11110:      #    ký tự 4 byte
        cnt = 3
    else:                        # 10xxxxxx hoặc 11111xxx làm byte đầu
        return False
return cnt == 0                  # ký tự cuối không được cụt
```

### Checklist

- [ ] Kiểm tra `cnt > 0` **trước** khi phân loại byte đầu.
- [ ] Đủ bốn tiền tố: `0` (>>7), `110` (>>5), `1110` (>>4), `11110` (>>3).
- [ ] Byte tiếp theo phải có `v >> 6 == 0b10`.
- [ ] Byte đầu dạng `10xxxxxx` hoặc `11111xxx` ⇒ `False`.
- [ ] Cuối cùng `return cnt == 0` (không trả `True` mù quáng).
- [ ] Test: `[197]`, `[145]`, `[255]`, `[240,144,128]`, `[0]`, mảng 20000 phần tử.